# Lab 02 · Retrieve architecture rules with RAG

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · 15–20 minute lab

## Retrieve the policy by meaning, not by matching words

**So far:** in Lab 01 we pasted the policy into every prompt.

**New:** retrieval-augmented generation (RAG) with **embeddings** and a **vector database**. We keep several policies in a store, find the ones that match the question, and give only that evidence to the model.

- An *embedding model* turns a text into a list of numbers, a *vector*. Texts with a similar meaning get vectors that point in a similar direction, even when they share no words.
- A *vector database* stores those vectors and quickly finds the ones nearest to the vector of a question.

We use the small `all-MiniLM-L6-v2` embedding model, which runs on the CPU, and the **Chroma** vector database in memory. Neither needs an API key. The first run downloads the model (about 80 MB) and shows a progress bar.

## Pattern · Retrieval-Augmented Generation

*Single-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Context** | Useful answers often depend on private, recent, or domain-specific material that the model's weights do not reliably hold. |
| **Problem** | Without evidence, a model may hallucinate, miss local facts, or answer from outdated knowledge. |
| **Solution** | Search a knowledge base for relevant pieces, place them beside the question, and instruct the model to ground its answer in them. |
| **Use when** | Answers depend on a document collection; citations or provenance matter; knowledge changes independently of the model. |
| **Trade-off** | Answer quality now depends on retrieval quality, chunking, and the context budget. |
| **Flow** | Question → Retrieve → Augment → Grounded answer |
| **Quality attributes** | Benefits: Reliability, Data freshness, Modifiability, Cost, Privacy. Liabilities: Latency, Maintainability, Reliability, Data freshness. |
| **In this lab** | Embeddings and a Chroma vector database retrieve the policies, a distance threshold makes the system abstain when nothing is close, and a citation check verifies that the answer is grounded. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'chromadb']):
    %pip install -q "agentic-ai-coursekit[rag] @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Store a few policies, and see where word matching fails

Only R1 is about our architecture question. The other policies are there so that retrieval has something to reject.

The question below is clearly about R1, but it is worded differently. `words` turns a text into a *set* of lower-case words, and `&` keeps the words that two sets share. Word matching sees only "services" and "the", and every policy contains "the".

In [ ]:
policies = {
    "R1": "Choose microservices only when the team has 8+ people and services must deploy independently; "
          "otherwise choose a modular monolith.",
    "R2": "Each module owns its tables. One module never reads the tables of another module.",
    "R3": "Deploy to the eu-west region unless a customer contract names another region.",
    "R4": "Public APIs keep the old version running for 6 months after a breaking change.",
}

def words(text):
    """The lower-case words in a text, without punctuation."""
    return {word.strip(".,;:?!").lower() for word in text.split()}

question = "How many engineers do we need before splitting the system into separate services?"
for pid, text in policies.items():
    print(pid, "shares", sorted(words(question) & words(text)))

## Step 2 · Turn text into vectors

`embed` takes a list of texts and returns one vector per text. Each vector has 384 numbers. On their own the numbers mean nothing to us; what matters is how two vectors compare.

*Cosine similarity* measures the angle between two vectors. A value near 1 means the texts point the same way, so they have a similar meaning. A value near 0 means they are unrelated.

In [ ]:
import numpy as np
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

embed = DefaultEmbeddingFunction()  # all-MiniLM-L6-v2, runs locally on the CPU
q_vec, r1_vec, r3_vec = embed([question, policies["R1"], policies["R3"]])
print("Each text becomes", len(q_vec), "numbers. The first four:", np.round(q_vec[:4], 3))

def cosine(a, b):
    """Near 1.0: similar meaning. Near 0: unrelated."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("question vs R1 (architecture):", round(cosine(q_vec, r1_vec), 2))
print("question vs R3 (deploy region):", round(cosine(q_vec, r3_vec), 2))

## Step 3 · Put the vectors in a vector database

A Chroma *collection* is like a table of documents. We give it the embedding function, so Chroma embeds each policy when we add it and embeds each question when we query. `upsert` inserts a policy, or updates it if the id already exists, so you can run this cell again safely.

`query` returns the nearest policies with their *cosine distance*, which is 1 minus the cosine similarity. A **smaller distance means a closer match**. The results come back as lists inside a list, because one query can ask several questions at once. `[0]` picks the answers for our one question.

In [ ]:
import chromadb
from chromadb.config import Settings

db = chromadb.Client(Settings(anonymized_telemetry=False))  # in memory: nothing is saved to disk
collection = db.get_or_create_collection("policies", embedding_function=embed,
                                         configuration={"hnsw": {"space": "cosine"}})
collection.upsert(ids=list(policies), documents=list(policies.values()))

found = collection.query(query_texts=[question], n_results=4)
for pid, distance in zip(found["ids"][0], found["distances"][0]):
    print(f"{pid}  distance {distance:.2f}")

## Step 4 · Keep only the close matches

The database always returns the nearest policies, even for a question that none of them answers. `retrieve` keeps the top two, then drops any policy farther away than `max_distance`. The result is a dictionary from policy id to distance, best first. The lunch question shows why the threshold matters.

In [ ]:
def retrieve(question, max_distance=0.7, top_k=2):
    """The nearest policies as {id: distance}, best first. Distant ones are dropped."""
    found = collection.query(query_texts=[question], n_results=top_k)
    pairs = zip(found["ids"][0], found["distances"][0])
    return {pid: round(distance, 2) for pid, distance in pairs if distance <= max_distance}

questions = [
    question,
    "Can the billing module read the tables of the orders module?",
    "What should the team eat for lunch on Friday?",
]
for q in questions:
    print(f"{str(retrieve(q)):<14} {q}")

## Step 5 · Answer from the evidence and check the citations

The prompt holds only the retrieved policies, each labelled with its id, and the model must cite an id like `[R1]`. Our code runs two checks around the model call:

- When nothing was retrieved, the model is **not called**. The program abstains instead of letting the model guess.
- `re.findall(r"R\d+", answer)` finds the cited ids. An answer is *grounded* when it cites at least one id and every cited id was retrieved. In Python, `cited <= set(hits)` asks "is `cited` a subset of the retrieved ids?".

In [ ]:
import re

scripted = ["At 8 or more people, and only if the services must deploy independently [R1].",
            "No. Each module owns its tables, so billing must ask the orders module [R2].",
            ""]  # the lunch question never reaches the model
verdicts = []  # one True/False per answered question
for q, example in zip(questions, scripted):
    hits = retrieve(q)
    if not hits:
        print(f"Q: {q}\nA: No matching policy, so the model is not called.\n")
        continue
    evidence = "\n".join(f"[{pid}] {policies[pid]}" for pid in hits)
    prompt = f"Answer using only these policies:\n{evidence}\n\nQuestion: {q}\nAnswer in one sentence and cite the policy id."
    answer = ask(prompt, example=example)
    cited = set(re.findall(r"R\d+", answer))
    verdicts.append(bool(cited) and cited <= set(hits))
    print(f"Q: {q}\nA: {answer}\nCited {sorted(cited)}, retrieved {list(hits)}, grounded: {verdicts[-1]}\n")
assert all(verdicts), "Each answer should cite only the policies it was given."

## What you should see

Step 1 shows that the question shares only "services" and "the" with R1, and "the" with every other policy. In Step 2, the question is much more similar to R1 than to R3. In Step 3, R1 is nearest at a distance of about 0.52, and the other policies are about 0.8 or farther.

In Step 4, the engineering question retrieves `R1`, the billing question retrieves `R2`, and the lunch question retrieves nothing, because its nearest policy is about 0.83 away. In Step 5, both answers are `grounded: True`, and the lunch question is never sent to the model. Distances can differ slightly in the last decimal place between computers.

## Try it

1. Add the question `"Which cloud region should host our servers?"` to `questions`, and add a scripted reply that cites `[R3]` to `scripted`. Check with `words` how few words it shares with R3, then run Steps 4 and 5.
2. Change `max_distance` to `0.4` and run Steps 4 and 5. Now nothing is retrieved, so nothing is answered. Then try `0.9`: the lunch question retrieves R1 and reaches the model. Its empty scripted reply cites nothing, so the final check fails.
3. Change the first scripted reply to cite `[R4]`. R4 was never retrieved, so the answer is not grounded and the final check fails.

## Takeaway

Embeddings let retrieval match meaning instead of exact words, and a vector database finds the nearest evidence quickly. The database always returns *something*, so the distance threshold decides when to abstain. Test the retriever as carefully as the prompt, because it decides what the model sees. Lab 03 starts Lecture 2: it splits a harder decision into a chain of checked prompts.

## Read further

- [The original RAG paper](https://arxiv.org/abs/2005.11401)
- [Sentence-BERT, the idea behind sentence embeddings](https://arxiv.org/abs/1908.10084)
- [The all-MiniLM-L6-v2 model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
- [Chroma documentation](https://docs.trychroma.com/)